# `polartox.polarized_trees` — Polarized Trees Demo

Given an annotation dataset (real or synthetic), Polarized Trees recursively partitions annotators by demographic dimension to find the specific subgroups that are most polarized on a given text.

This notebook walks through the full pipeline (paper Steps 1–6), using the synthetic dataset from `polartox.datagen` so we have known ground truth to check against.

**Requires:** `pip install "polartox[ndfu]"` (uses the collaborative [`ndfu`](https://github.com/ipavlopoulos/ndfu) package for nDFU scoring).

In [ ]:
import numpy as np
import pandas as pd

from polartox.datagen import AnnotatorPool, DEFAULT_DIMENSIONS, DEFAULT_DEPTH_WEIGHTS, DEFAULT_INTENSITY_RANGE
from polartox.polarized_trees import PolarizedTreesPipeline, detect_polarized_subgroups, render_tree_text

## 1. Generate a dataset with known ground truth

Using `polartox.datagen` so every text's true active dimensions, lean, and intensity are known -- letting us check whether the tree actually recovers them.

In [ ]:
pool = AnnotatorPool(
    dimensions=DEFAULT_DIMENSIONS,
    scale=5,
    intensity_range=DEFAULT_INTENSITY_RANGE,
    depth_weights=DEFAULT_DEPTH_WEIGHTS,
    annotators_per_identity=10,
)

result = pool.generate_dataset(n_texts=200, n_annotators_per_text=None, noise=0.05, seed=42)
dataset, ground_truth = result
print(f"Dataset shape: {dataset.shape}")

## 2. Build the pipeline

Key parameters, all empirically validated on this synthetic generator (see project notes -- these are deliberate choices, not the paper's literal defaults):

- `min_size_frac` (not the paper's fixed `min_size`): minimum subgroup size as a **fraction** of each text's own annotator count, so the same config works across datasets with very different annotator densities.
- `theta_stop` (not in the paper): stop a node immediately if its own nDFU is already low, without searching for a split. Prevents noise-driven over-fragmentation in small subgroups.
- `variant="beta"` (the paper names PRGmax as primary): PRGbeta, the harmonic mean of PRGmax and PRGvar, empirically outperformed either alone on this synthetic validation data.

In [ ]:
pipe = PolarizedTreesPipeline(
    dims=list(DEFAULT_DIMENSIONS.keys()),
    scale=pool.scale,
    theta_filter=0.3,      # Step 2 -- user-defined; how polarized a text must be to analyze at all
    min_size_frac=0.03,    # ~3% of each text's annotators per resulting subgroup
    h=0.15,                # PRG gain threshold -- interpreted as a FRACTION of remaining nDFU (relative_h=True)
    max_depth=6,
    variant="beta", beta=1.0,
    theta_stop=0.15,
    relative_h=True,       # recommended default -- see README "Departures from the paper"
)

## 3. Run the full evaluation

Runs Steps 1–6 end to end: filters unpolarized texts, builds one tree per remaining text, then computes dataset-level metrics. Passing `ground_truth` additionally unlocks recovery metrics (jaccard/precision/recall/exact-match) and enriches F/C/P with validation columns -- only meaningful because this is synthetic data with a known answer.

In [ ]:
results = pipe.run_full_evaluation(dataset, ground_truth=ground_truth)

## 4. Inspect one tree directly

`inspect_tree` prints the tree level by level, with the actual rating histogram at every node -- the same view used throughout development to hand-verify the algorithm's decisions.

In [ ]:
example_id = list(pipe.trees_.keys())[0]
print(f"Ground truth for text {example_id}: {ground_truth[example_id]}\n")
pipe.inspect_tree(example_id, dataset, show_distributions=True)

## 5. Step 6 metrics in detail

- **F (dimension frequency)**: how often each dimension is the splitting dimension, by depth. High frequency at depth 1 signals a primary driver; depth > 1 signals a secondary, intersectional role.
- **C (subgroup pole consistency)**: for each intersectional subgroup that appears as a leaf, how many trees it appears in (`n_s`) and how consistently it lands on the same pole.
- **P (subgroup PRG)**: the mean PRG of the split that produced each subgroup, averaged across every tree it appears in -- a measure of how strongly and consistently that split reduces polarization.

With `ground_truth` supplied, each table gains a validation column (`ever_truly_active`, `true_lean_match_rate`, `mean_true_alpha`) showing whether the metric's own conclusions line up with what was actually injected.

In [ ]:
print("=== F: dimension frequency by depth ===")
print(results["F"])

print("\n=== C: subgroup pole consistency (top 10 by occurrence) ===")
print(results["C"].head(10))

print("\n=== P: subgroup PRG (top 10 by mean PRG) ===")
print(results["P"].head(10))

## 6. Recovery, broken down by depth (k)

How well the tree recovers the true active dimensions, split by how many dimensions were actually injected. Recovery is typically strong at k=0 (correct negative control), k=1, k=3, and k=4; k=2 texts with two moderate-intensity causes are the hardest case -- a known, understood property of the generator's intensity mechanism rather than a tree-algorithm defect (see project notes for the diagnosis).

In [ ]:
recovery = results["recovery"]
print(recovery.groupby("k_true").agg(
    mean_jaccard=("jaccard", "mean"),
    mean_precision=("precision", "mean"),
    mean_recall=("recall", "mean"),
    exact_match_rate=("exact_match", "mean"),
    n=("text_id", "count"),
))

## 7. Using it without ground truth (real data)

On real annotation data (no injected answer to check against), `diagnostics()` and the plain F/C/P tables are the actual deliverable -- everything below works with zero knowledge of "the truth":

In [ ]:
pipe_no_gt = PolarizedTreesPipeline(
    dims=list(DEFAULT_DIMENSIONS.keys()), scale=pool.scale,
    theta_filter=0.3, min_size_frac=0.03, h=0.15, max_depth=6,
    relative_h=True,
)
results_no_gt = pipe_no_gt.run_full_evaluation(dataset, ground_truth=None)
# results_no_gt has "F", "C", "P", "diagnostics" -- no "recovery" key, since there's
# no ground truth to check against.